In [12]:
import numpy as np
import pandas as pd
import os

rng = np.random.default_rng(42)

N_DEV = 25_000
N_OOT = 6_000

PRODUCTS = ["personal", "auto", "mortgage", "SME"]
EMPLOYMENT = ["salaried", "self_employed", "professional"]
REGIONS = ["metro", "tier1", "tier2", "tier3"]


def generate(n, oot=False):
    """Synthetic origination-level data. `oot=True` applies mild drift."""
    age = np.clip(rng.normal(38, 10, n), 21, 65).round(0)

    log_income = rng.normal(12.8, 0.45, n)
    if oot:  # income stress in later vintages
        log_income = log_income - 0.06
    annual_income = np.exp(log_income).round(-3)

    bureau_mean = 700 if not oot else 690  # bureau quality drifts down
    bureau_score = np.clip(rng.normal(bureau_mean, 80, n), 300, 900).round(0)

    product = rng.choice(PRODUCTS, n, p=[0.35, 0.20, 0.20, 0.25])
    employment = rng.choice(EMPLOYMENT, n, p=[0.55, 0.30, 0.15])
    region = rng.choice(REGIONS, n, p=[0.4, 0.25, 0.2, 0.15])

    # loan size scales with income; mortgages and SME are larger tickets
    size_mult = np.where(product == "mortgage", 4.0,
                np.where(product == "SME", 2.5, 1.0))
    loan_amount = (annual_income * size_mult * rng.uniform(0.5, 1.5, n)).round(-4)
    tenure_months = np.where(product == "mortgage", rng.integers(120, 300, n),
                      np.where(product == "auto", rng.integers(36, 84, n),
                      np.where(product == "SME", rng.integers(48, 120, n),
                               rng.integers(12, 60, n))))

    emi = loan_amount / tenure_months * 1.4  # rough annuity at ~14% p.a.
    other_emi = annual_income / 12 * rng.uniform(0.05, 0.35, n)
    dti_ratio = np.clip((emi + other_emi) / (annual_income / 12), 0, 3).round(3)

    delinq_24m = rng.poisson(0.25, n)
    months_since_delinq = np.where(delinq_24m > 0,
                                   rng.integers(1, 24, n), 99)

    # ---- latent default data-generating process (12-month PD) ----
    logit = (-3.90
             + 0.95 * (680 - bureau_score) / 80
             + 1.30 * (dti_ratio - 0.45)
             - 0.70 * (log_income - 12.8) / 0.45
             + 0.45 * (employment == "self_employed")
             + 0.40 * (product == "SME")
             - 0.30 * (product == "mortgage")
             + 0.40 * np.minimum(delinq_24m, 3)
             + 0.25 * (region == "tier3"))
    pd_true = 1 / (1 + np.exp(-logit))
    default_12m = rng.binomial(1, pd_true)

    df = pd.DataFrame({
        "age": age,
        "annual_income": annual_income,
        "bureau_score": bureau_score,
        "product_type": product,
        "employment_type": employment,
        "region": region,
        "loan_amount": loan_amount,
        "tenure_months": tenure_months,
        "dti_ratio": dti_ratio,
        "delinq_24m": delinq_24m,
        "months_since_delinq": months_since_delinq,
        "pd_true": pd_true.round(5),
        "default_12m": default_12m,
        # outstanding balance as EAD proxy (70-100% of origination)
        "ead": (loan_amount * rng.uniform(0.70, 1.0, n)).round(-3),
        "sample": "oot" if oot else "dev",
    })
    return df


dev = generate(N_DEV, oot=False)
oot = generate(N_OOT, oot=True)

os.makedirs("data", exist_ok=True) # Create the directory if it doesn't exist
dev.to_csv("data/portfolio_dev.csv", index=False)
oot.to_csv("data/portfolio_oot.csv", index=False)

print(f"Development sample : {len(dev):,} rows | default rate {dev['default_12m'].mean():.2%}")
print(f"Out-of-time sample : {len(oot):,} rows | default rate {oot['default_12m'].mean():.2%}")
print("\nSegment default rates (dev):")
print(dev.groupby("product_type")["default_12m"].agg(["count", "mean"]).round(4))

Development sample : 25,000 rows | default rate 5.49%
Out-of-time sample : 6,000 rows | default rate 7.02%

Segment default rates (dev):
              count    mean
product_type               
SME            6248  0.0775
auto           4987  0.0419
mortgage       5047  0.0321
personal       8718  0.0594
